In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

x = torch.rand(2, 20)
print(x)
y = net(x)
print(y)

tensor([[6.2345e-02, 4.4062e-02, 1.1032e-01, 6.7037e-01, 1.5805e-01, 8.6349e-01,
         5.6697e-01, 6.2824e-01, 7.6957e-01, 6.5435e-01, 3.9608e-01, 2.2396e-01,
         4.2472e-01, 5.4605e-01, 1.1677e-01, 7.2362e-01, 9.1705e-01, 9.1385e-01,
         8.7600e-01, 9.8849e-01],
        [4.8232e-04, 5.3764e-01, 3.9078e-01, 9.1691e-01, 7.0618e-01, 8.7893e-01,
         4.1253e-01, 1.1562e-01, 4.4182e-01, 3.1785e-01, 4.6193e-01, 9.9268e-01,
         5.9462e-01, 9.0701e-01, 3.5176e-02, 5.0168e-01, 1.3928e-01, 3.2087e-01,
         8.2710e-01, 9.0449e-01]])
tensor([[-0.1420,  0.3618,  0.3342,  0.2994, -0.0594, -0.0548,  0.1657,  0.2810,
         -0.0840,  0.0046],
        [-0.0994,  0.3468,  0.1754,  0.2558, -0.0218,  0.0269,  0.0271,  0.3025,
         -0.0611,  0.0886]], grad_fn=<AddmmBackward0>)


In [4]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, x):
        x = self.hidden(x)
        x = F.relu(x)
        x = self.out(x)
        return x

In [8]:
net = MLP()
net(x)

tensor([[ 0.0998, -0.0655,  0.0471, -0.1951, -0.0274,  0.3280, -0.2238, -0.0846,
         -0.2255,  0.0061],
        [ 0.0144, -0.0822,  0.0052,  0.0275,  0.0815,  0.2111, -0.1476, -0.0594,
         -0.2135, -0.0046]], grad_fn=<AddmmBackward0>)

In [9]:
# Sequential_block

class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X

In [18]:
net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(x)

tensor([[ 2.1158e-01,  1.9883e-01, -2.5413e-01, -1.6168e-01, -1.7171e-01,
          1.1347e-01, -1.8052e-02, -6.3512e-02,  2.1255e-01, -5.7179e-03],
        [ 2.6114e-01,  1.0617e-01, -3.2064e-01, -2.9649e-01, -1.6408e-01,
          5.2593e-02,  2.5412e-04, -9.8600e-02,  1.6502e-01,  7.8936e-02]],
       grad_fn=<AddmmBackward0>)

In [13]:
# demo
def example(*args):
    for arg in args:
        print(arg)
example(1, 2, 3)

1
2
3


In [16]:
# demo 
# enumerate() returns an iterator
def Myenumerate(*args):
    for idx ,module in enumerate(args):
        print(idx, module)
Myenumerate(nn.Linear(10, 5), nn.ReLU())

0 Linear(in_features=10, out_features=5, bias=True)
1 ReLU()


In [26]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((40, 20), requires_grad = False)
        self.linear_haha = nn.Linear(20, 40)

    def forward(self, x):  
        x = self.linear_haha(x)
        x = torch.mm(x, self.rand_weight) + 1
        x = F.relu(x)
        x = self.linear_haha(x)
        while x.abs().sum() > 1:
            x /= 2
        return x.sum()

In [27]:
net = FixedHiddenMLP()
net(x)

tensor(-0.0883, grad_fn=<SumBackward0>)

In [28]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64),
                                nn.ReLU(),
                                nn.Linear(64, 32),
                                nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        X = self.net(X)
        X = self.linear(X)
        return X

In [30]:
chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(x)

tensor(0.0627, grad_fn=<SumBackward0>)